# Membrane-Based CO2 Separation

This notebook explores membrane-based gas separation for CO2 capture. Membranes offer advantages over absorption:
- No solvent regeneration needed
- Modular and scalable
- Lower energy for moderate purity requirements

## Learning Objectives

1. Understand the solution-diffusion model for gas permeation
2. Compare different membrane materials
3. Design single and multi-stage membrane systems
4. Analyze trade-offs between recovery, purity, and area

## 1. Background: Membrane Gas Separation

### The Solution-Diffusion Model

Gas permeates through dense polymer membranes by:
1. **Dissolving** into the membrane at the high-pressure side
2. **Diffusing** through the membrane material
3. **Desorbing** at the low-pressure side

The flux of component $i$ is:

$$J_i = \frac{P_i}{\delta}(p_{i,feed} - p_{i,permeate})$$

where:
- $P_i$ = permeability (Barrer = 10⁻¹⁰ cm³(STP)·cm/(cm²·s·cmHg))
- $\delta$ = membrane thickness
- $p_i$ = partial pressure of component $i$

### Selectivity

The selectivity determines separation quality:

$$\alpha_{i/j} = \frac{P_i}{P_j}$$

For CO2/N2 separation:
- Commercial polymers: α = 20-40
- High-performance materials: α = 40-100+

### Robeson Upper Bound

There's a fundamental trade-off: high permeability membranes tend to have low selectivity. The Robeson upper bound (2008) defines the empirical limit:

$$P_{CO2} = k \cdot \alpha^{-n}$$

## 2. Setup

In [ ]:
import jax
import jax.numpy as jnp

jax.config.update("jax_enable_x64", True)

from difflow.streams import make_stream, get_flows, total_flow

from difflow_cc import (
    get_membrane, list_membranes,
    MembraneParams, MembraneSeparator,
)
from difflow_cc.units.membrane import MultistageMembrane

print("Available membranes:", list_membranes())

## 3. Exploring the Membrane Database

In [ ]:
# Get Matrimid properties (common commercial polyimide)
matrimid = get_membrane("Matrimid")

print(f"Membrane: {matrimid.full_name}")
print(f"Type: {matrimid.membrane_type}")
print()
print("Permeability (Barrer):")
for gas, perm in matrimid.permeability.items():
    print(f"  {gas}: {perm}")
print()
print("Selectivity:")
for pair, sel in matrimid.selectivity.items():
    print(f"  {pair}: {sel}")
print()
print(f"Typical thickness: {matrimid.typical_thickness} μm")
print(f"Max temperature: {matrimid.max_temperature - 273.15:.0f} °C")
print(f"Cost: ${matrimid.cost_usd_m2}/m²")

In [ ]:
# Compare all membrane materials
print(f"{'Membrane':<18} {'Type':<12} {'P_CO2 (Barrer)':<16} {'α (CO2/N2)':<12}")
print("-" * 58)

for mem_name in list_membranes():
    mem = get_membrane(mem_name)
    p_co2 = mem.permeability.get('CO2', 0)
    alpha = mem.selectivity.get('CO2_N2', mem.selectivity.get('CO2_CH4', 0))
    print(f"{mem_name:<18} {mem.membrane_type:<12} {p_co2:<16.1f} {alpha:<12.1f}")

## 4. Single-Stage Membrane Separation

Key design parameters:
- **Area**: Larger area → higher recovery, but more cost
- **Pressure ratio**: Higher ratio → better driving force
- **Stage cut**: Fraction of feed that permeates (θ = F_permeate/F_feed)

In [ ]:
# Define feed gas (flue gas at elevated pressure)
feed = make_stream(
    flows={"CO2": 1.5, "N2": 8.5},  # 15% CO2
    T=298.15,  # 25°C
    P=1000000.0,  # 10 bar
)

print("Feed Gas:")
print(f"  Total flow: {total_flow(feed):.1f} mol/s")
print(f"  CO2: {float(feed['F_CO2']):.1f} mol/s ({float(feed['F_CO2']/total_flow(feed)):.0%})")
print(f"  Pressure: {float(feed['P'])/1e5:.1f} bar")

In [ ]:
# Configure membrane separator
params = MembraneParams(
    membrane_type="Matrimid",
    area=1000.0,  # m²
    pressure_ratio=10.0,  # Feed/permeate pressure
    feed_pressure=1000000.0,  # 10 bar
    T_operation=298.15,
)

membrane = MembraneSeparator(params)

# Run separation
retentate, permeate, info = membrane(feed)

print("Membrane Separation Results:")
print(f"  Stage cut: {float(info['stage_cut']):.2%}")
print(f"  CO2 recovery: {float(info['CO2_recovery']):.1%}")
print(f"  CO2 purity in permeate: {float(info['CO2_purity']):.1%}")
print()
print("Permeate (CO2-enriched):")
perm_flows = get_flows(permeate)
print(f"  Flow: {float(info['permeate_flow']):.3f} mol/s")
print(f"  Pressure: {float(permeate['P'])/1e5:.1f} bar")
print()
print("Retentate (treated gas):")
ret_flows = get_flows(retentate)
print(f"  Flow: {float(info['retentate_flow']):.3f} mol/s")
print(f"  CO2 remaining: {float(ret_flows['CO2']):.3f} mol/s")

## 5. Effect of Membrane Area

Larger membrane area increases recovery but has diminishing returns.

In [ ]:
areas = [100, 250, 500, 1000, 2000, 5000]

print(f"{'Area (m²)':<12} {'Stage Cut':<12} {'CO2 Recovery':<14} {'CO2 Purity':<12}")
print("-" * 50)

for area in areas:
    params = MembraneParams(
        membrane_type="Matrimid",
        area=float(area),
        pressure_ratio=10.0,
        feed_pressure=1000000.0,
    )
    membrane = MembraneSeparator(params)
    _, _, info = membrane(feed)
    
    print(f"{area:<12} {float(info['stage_cut']):.2%}        "
          f"{float(info['CO2_recovery']):.1%}          "
          f"{float(info['CO2_purity']):.1%}")

## 6. Comparing Membrane Materials

In [ ]:
membranes = ["Matrimid", "PDMS", "CA", "PIM1", "ZIF8_Matrimid"]

print(f"{'Membrane':<18} {'Stage Cut':<12} {'CO2 Recovery':<14} {'CO2 Purity':<12}")
print("-" * 56)

for mem_name in membranes:
    try:
        params = MembraneParams(
            membrane_type=mem_name,
            area=1000.0,
            pressure_ratio=10.0,
            feed_pressure=1000000.0,
        )
        membrane = MembraneSeparator(params)
        _, _, info = membrane(feed)
        
        print(f"{mem_name:<18} {float(info['stage_cut']):.2%}        "
              f"{float(info['CO2_recovery']):.1%}          "
              f"{float(info['CO2_purity']):.1%}")
    except Exception as e:
        print(f"{mem_name:<18} Error: {e}")

## 7. Multi-Stage Membrane Cascades

Single-stage membranes face a trade-off: high recovery means low purity.

Multi-stage configurations can achieve both:
- **Series**: Each stage treats the retentate from the previous
- **Permeate recycle**: Second stage enriches the permeate further

In [ ]:
# Single stage baseline
single_params = MembraneParams(
    membrane_type="Matrimid",
    area=500.0,
    pressure_ratio=10.0,
    feed_pressure=1000000.0,
)
single = MembraneSeparator(single_params)
_, _, single_info = single(feed)

# Two-stage series
series_cascade = MultistageMembrane(
    single_params,
    n_stages=2,
    configuration="series"
)
_, _, series_info = series_cascade(feed)

# Two-stage permeate recycle
recycle_cascade = MultistageMembrane(
    single_params,
    n_stages=2,
    configuration="permeate_recycle"
)
_, _, recycle_info = recycle_cascade(feed)

print(f"{'Configuration':<25} {'CO2 Recovery':<15} {'CO2 Purity':<15}")
print("-" * 55)
print(f"{'Single stage (500 m²)':<25} {float(single_info['CO2_recovery']):.1%}          "
      f"{float(single_info['CO2_purity']):.1%}")
print(f"{'Two-stage series':<25} {float(series_info['overall_CO2_recovery']):.1%}          "
      f"{float(series_info['overall_CO2_purity']):.1%}")
print(f"{'Two-stage w/ recycle':<25} {float(recycle_info['overall_CO2_recovery']):.1%}          "
      f"{float(recycle_info['overall_CO2_purity']):.1%}")

## 8. Key Takeaways

1. **Membrane separation** is governed by permeability and selectivity

2. **Trade-offs exist**:
   - Recovery vs purity (single stage)
   - Permeability vs selectivity (Robeson bound)
   - Area vs cost

3. **Material selection** depends on application:
   - PDMS: High flux, moderate selectivity (natural gas)
   - Polyimides: Good selectivity (post-combustion)
   - Mixed-matrix: Approaching upper bound

4. **Multi-stage configurations** overcome single-stage limitations